# 단계 6 — 클로드 코드에 등록하고 실전에서 사용하기 (보너스 단계)

**`structural-mcp` 누적 빌드업의 단계 ⑥**: 단계 5에서 완성한 서버를 클로드 코드 명령행 인터페이스에 등록하여 실제 자연어 워크플로를 체험합니다.

## 본 노트북의 위치
이 노트북은 강의노트 7주차 본문 §2.7 단계 ⑥ 라인 1574 부근에 명시된 클로드 코드 등록 단계를 그대로 구현합니다. 단계 1부터 단계 5까지 누적된 완성형 서버를 처음으로 외부 도구에 연결하여 자연어로 호출해 보는 단계이며, 본 강의 7주차 실습의 마지막 단계에 해당합니다.

## 학습 목표
이번 단계에서는 클로드 코드 명령행 인터페이스에 사용자가 직접 만든 모델 컨텍스트 프로토콜 서버를 **단 한 줄 명령**으로 등록하는 방법을 익힙니다. 등록된 서버를 모든 클로드 코드 세션에서 자연어로 호출할 수 있음을 실제로 확인하고, 9주차 멀티 에이전트 패턴에서 본 서버가 한 명의 도메인 에이전트로 차지할 역할을 머릿속에 그려 봅니다. 본 단계의 가장 큰 가치는 *학생과 실무자가 직접 만든 서버가 어떻게 강의 후반부 전반에 걸쳐 재사용되는지*를 한눈에 체험할 수 있다는 점에 있습니다.

## 선행 학습 사항
단계 5까지 완성된 `structural_mcp.py` 파일이 손에 있어야 합니다. 도구 세 개와 리소스 한 개와 프롬프트 두 개가 등록되어 있어야 합니다. 클로드 코드 명령행 인터페이스가 설치되어 있어야 하며, 보통 `npm i -g @anthropic-ai/claude-code` 명령으로 설치합니다. 패키지 매니저로는 `uv`를 권장하지만 미설치 시에는 표준 파이썬 인터프리터로 대체할 수 있습니다.

## 본 단계 이후의 흐름
본 노트북을 마치고 나면 7주차의 모든 실습이 마무리됩니다. 8주차에서는 클로드 코드를 더 깊이 활용하는 서브에이전트와 후크와 깃 워크트리 등을 다루고, 9주차에서는 본 단계에서 등록한 서버가 멀티 에이전트의 한 명으로 동작하는 라우팅 패턴을 익히며, 11주차에서는 본 노트북의 미다스 파서를 실제 해석 결과 처리까지 확장하는 응용을 다룹니다.


## §1. 클로드 코드의 모델 컨텍스트 프로토콜 통합 개요

클로드 코드는 등록된 모델 컨텍스트 프로토콜 서버를 **외부 도구로 자동 연결**합니다. 한 번 등록만 해두면 다양한 자연어 워크플로가 모두 가능해집니다. 이 단계는 그동안 만든 서버가 학습용 노트북을 벗어나 실제 사용자의 일상 작업 환경에 들어가는 첫 순간이라는 점에서 의미가 큽니다.

예컨대 사용자가 *"샘플 모델 파일의 부재 개수를 알려줘"* 라고 말하면 클로드 코드는 자동으로 미다스 파서 도구를 호출합니다. *"폭 300, 깊이 600 보의 휨을 검토해줘"* 라고 말하면 구조 검토 프롬프트가 발동하면서 휨 검토 도구가 따라서 호출됩니다. *"KDS 4.3 조항을 보여줘"* 라고 말하면 KDS 요약 리소스가 자동으로 조회됩니다. 이러한 자연어 워크플로는 사용자가 명령행 도구의 정확한 사용법을 외울 필요가 없게 만들어 주며, 도메인 전문가가 아니더라도 본 서버의 능력을 활용할 수 있게 해 줍니다.

한 번 등록하면 **모든 프로젝트에서 재사용** 가능합니다. 이는 강의노트 §2.7 라인 1903부터 1916에서 강조하는 **"클로드 코드는 모델 컨텍스트 프로토콜의 소비자 측면을 대표한다"** 는 메시지를 직접 체험하는 단계입니다.

```mermaid
graph LR
    subgraph 세션["클로드 코드 세션"]
        사용자 --> 클로드
    end
    subgraph 서버목록["등록된 모델 컨텍스트 프로토콜 서버들"]
        S1["structural-mcp 본 노트북에서 직접 만든 서버"]
        S2["filesystem 기본 제공"]
        S3["git 기본 제공"]
    end
    클로드 <-->|stdio 전송| S1
    클로드 <-->|stdio 전송| S2
    클로드 <-->|stdio 전송| S3
    style S1 fill:#e8f4f8,stroke:#2980b9
```

> [!tip] 8주차·9주차·11주차로 이어지는 연결고리
> 본 단계에서 등록한 `structural-mcp`는 **8주차의 클로드 코드 심화**, **9주차의 멀티 에이전트 라우팅**, **11주차의 미다스 모델 컨텍스트 프로토콜 응용** 단계에서 모두 재사용됩니다. 즉 한 번 만들어 두면 강의 후반부 전반에 걸쳐 같은 도메인 능력을 활용할 수 있으며, 학습자가 졸업 후 실무에 진입한 뒤에도 자신만의 전문 서버로 평생 발전시켜 갈 수 있습니다.


## §2. 사전 점검 — 명령행 인터페이스 설치 확인

`claude` 명령이 운영체제의 실행 경로(PATH)에 등록되어 있는지, 그리고 `uv` 패키지 매니저나 표준 파이썬 인터프리터 중 적어도 하나가 사용 가능한지 확인합니다. 명령행 인터페이스가 미설치된 경우 본 노트북은 등록 명령을 화면에 출력만 할 뿐, 실제 등록은 건너뜁니다.

본 셀에서 수행하는 점검은 등록 단계를 진행하기에 앞서 환경 준비가 되어 있는지 확인하는 안전 장치 역할을 합니다. 만약 어느 한 가지라도 빠져 있다면 본 셀이 친절한 안내 문구를 출력해 주므로, 학습자는 어떤 도구를 추가로 설치해야 하는지 한눈에 파악할 수 있습니다.

In [ ]:
# Week_07.md §2.7 단계 ⑥ — claude CLI 가용 여부 확인
import subprocess
import shutil
from pathlib import Path

claude_cmd = shutil.which("claude")
if claude_cmd is None:
    print("[!] claude CLI not found on PATH.")
    print("    Install guide: 02-Resources/Guide-Claude-CLI-Setup.md")
    print("    npm i -g @anthropic-ai/claude-code")
else:
    print(f"[OK] claude found at: {claude_cmd}")
    try:
        out = subprocess.run(["claude", "--version"], capture_output=True, text=True, timeout=5)
        print(f"     version: {out.stdout.strip() or out.stderr.strip()}")
    except Exception as e:
        print(f"     (version check failed: {e})")

uv_cmd = shutil.which("uv")
print(f"\n[uv]    found: {uv_cmd or '(not installed — `pip install uv`)'}")

py_cmd = shutil.which("python") or shutil.which("python3")
print(f"[python] found: {py_cmd}")


## §3. 본 서버를 등록하는 명령

클로드 코드의 등록 명령은 다음과 같은 일반 형식을 갖습니다. 서버 이름과 실행 명령과 인자들을 차례로 적어 줍니다.

```bash
claude mcp add <서버 이름> -- <실행 명령> <인자들...>
```

본 서버를 등록하는 경우에는 두 가지 변형이 있습니다. 첫 번째는 `uv` 패키지 매니저를 활용하는 권장 방식으로, 서버 이름 뒤에 `uv run` 명령과 본 노트북에서 만든 통합 파일의 절대경로를 함께 적습니다. 두 번째는 `uv`가 설치되지 않은 환경에서 사용하는 대체 방식으로, `uv run` 자리에 `python` 명령을 사용합니다. 두 방식 모두 동일한 결과를 만들어내므로 환경에 맞춰 선택하면 됩니다.

> [!tip] 절대경로 사용은 필수입니다
> 클로드 코드는 등록된 서버를 임의의 디렉터리에서 실행하므로 **상대경로는 동작하지 않습니다**. 반드시 `Path.resolve()`로 산출한 절대경로를 등록 명령에 사용해야 합니다. 이는 운영 규칙의 'absolute paths only' 원칙과도 일치하며, 학생들이 자주 부딪히는 실수 중 하나이므로 처음부터 절대경로를 사용하는 습관을 들이는 것이 좋습니다.

> [!finding] 등록 범위 — 프로젝트별 vs 전역
> 사용자 홈 디렉터리에 있는 전역 설정 파일에 등록하면 모든 프로젝트에서 재사용할 수 있습니다. 본인 컴퓨터에서 자주 쓰는 서버라면 이 방식을 권장합니다. 한편 프로젝트 루트에 있는 프로젝트별 설정 파일에 등록하면 해당 프로젝트에서만 자동으로 로드됩니다. 팀과 깃으로 공유할 때는 이 방식이 적합하며, 학생들이 팀 프로젝트를 진행할 때 가장 자주 사용하는 방식이기도 합니다.


In [ ]:
# Week_07.md §2.7 단계 ⑥ — 등록 명령 미리보기 (실제 실행은 터미널 권장)
import os
from pathlib import Path

py_path = Path("structural_mcp.py").resolve()
if not py_path.exists():
    print(f"[!] {py_path} not found.")
    print("    Run S6_st05 (Stage 5) first to generate structural_mcp.py.")
else:
    print(f"[OK] structural_mcp.py at: {py_path}  ({py_path.stat().st_size} bytes)")

print("\n=== 등록 명령 (터미널에서 직접 실행 권장) ===\n")
print(f"  claude mcp add structural-mcp -- uv run {py_path}")
print("\n  # uv 미설치 시 대체:")
print(f"  claude mcp add structural-mcp -- python {py_path}")

print("\n[Note] 노트북에서 자동 실행을 원하면 다음 셀의 주석을 해제하세요.")
print("       단, claude CLI가 설치돼 있어야 합니다.")


In [ ]:
# (선택) 노트북에서 직접 등록 — 주석 해제 후 실행
# import subprocess
# from pathlib import Path
# py_path = str(Path("structural_mcp.py").resolve())
# result = subprocess.run(
#     ["claude", "mcp", "add", "structural-mcp", "--", "uv", "run", py_path],
#     capture_output=True, text=True,
# )
# print("STDOUT:", result.stdout)
# print("STDERR:", result.stderr)
# print("return code:", result.returncode)
print("(commented — uncomment to run actual registration)")


## §4. 등록 검증하기 — 등록 목록 조회 명령

등록이 정상적으로 끝났는지 확인하기 위해 `claude mcp list` 명령을 사용합니다. 출력 결과에 본 서버의 이름이 보이면 성공입니다. 만약 보이지 않으면 §3의 등록 명령을 다시 실행해 봅니다.

이 명령은 단순히 본 서버의 등록 여부만 확인하는 것이 아니라, 같은 클로드 코드 환경에 등록된 다른 서버들도 모두 보여 줍니다. 즉 본인이 사용 중인 가상 연구실의 구성원이 누구인지 한 화면에서 확인할 수 있습니다.

In [ ]:
# 등록된 MCP 서버 목록 확인
import subprocess
import shutil

if shutil.which("claude"):
    try:
        result = subprocess.run(["claude", "mcp", "list"],
                                capture_output=True, text=True, timeout=10)
        print("=== Registered MCP servers ===")
        print(result.stdout or "(empty list)")
        if "structural-mcp" in result.stdout:
            print("\n[✓] structural-mcp registered successfully.")
        else:
            print("\n[!] structural-mcp not yet registered. Run §3 first.")
    except Exception as e:
        print(f"[err] {e}")
else:
    print("[skip] claude CLI not installed.")


## §5. 클로드 코드 세션에서 자연어로 사용하기

이제 새 터미널을 열어 `claude` 명령으로 클로드 코드 세션을 시작한 뒤, 다음과 같이 자연어로 질의해 봅니다.

### 예시 1 — 직접 검토를 요청하는 경우
사용자가 다음과 같이 질의한다고 가정해 봅시다.

> 폭 300, 유효 깊이 540 의 철근콘크리트 보, 콘크리트 압축강도 27 메가파스칼, 철근 항복강도 400 메가파스칼, 인장 철근 단면적 1963 제곱밀리미터(이는 25 밀리미터 직경 철근 네 개에 해당), 소요 휨 모멘트 200 킬로뉴턴 미터. KDS 41 17 00 기준으로 휨을 검토해 주세요.

이 요청에 대한 클로드 코드의 예상 동작 순서는 다음과 같습니다. 첫째로 본 서버에 도구 목록을 자동으로 조회합니다. 둘째로 KDS 요약 리소스를 자동으로 가져와 강도감소계수가 0.85 임을 확인합니다. 셋째로 휨 검토 도구를 호출하여 수치 결과를 받습니다. 그러면 공칭 휨강도가 약 380 킬로뉴턴 미터, 소요/공칭 강도비가 0.53 으로 만족 판정을 받습니다. 마지막으로 KDS 4.3.1 조항을 인용하는 형식으로 한국어 답변을 생성합니다. 사용자는 그 어떤 도구의 정확한 인자명도 외울 필요 없이 한국어로 자연스럽게 요청만 하면 됩니다.

### 예시 2 — 프롬프트 카탈로그를 활용하는 경우
클로드 코드 세션에서 슬래시 메뉴를 누르면 **본 서버가 노출한 프롬프트가 자동으로 표시**됩니다. 보·기둥·슬래브 일반 검토용 프롬프트와 기둥 축력-휨 상호작용 전용 프롬프트가 한꺼번에 보입니다. 사용자가 프롬프트를 선택하고 인자만 채우면 검토가 바로 시작됩니다. 긴 도메인 명령을 외울 필요가 사라지며, 학생이든 실무자이든 누구나 같은 형식의 검토 결과를 받게 됩니다.

### 예시 3 — 미다스 모델 분석을 요청하는 경우
사용자가 *"내 프로젝트에 있는 어떤 모델 파일의 부재 수와 사용된 콘크리트 강도를 알려줘"* 라고 절대경로와 함께 질의하면, 클로드 코드는 자동으로 미다스 파서 도구를 호출하여 JSON 요약을 받고, 그 결과를 한국어로 풀어 답변합니다. 한국 구조설계 사무소에서 자주 만나는 시나리오이며, 본 서버의 가장 실용적인 활용 사례 중 하나입니다.


## §6. 등록 해제 (필요할 때)

시험이 끝난 뒤 정리하고 싶거나, 본 노트북에서 만든 통합 파일을 크게 수정하여 다시 등록하고 싶을 때는 등록 해제 명령을 사용합니다. 일반적으로 도구의 시그니처가 변경되었거나 새로운 도구를 추가한 경우라면 등록을 해제하고 다시 등록하는 것이 가장 안전한 방법입니다.

특히 본 강의를 마치고 본인의 도메인 서버를 더 다듬어 가는 과정에서는 등록과 해제를 여러 번 반복하게 됩니다. 등록 해제 명령을 자연스럽게 사용할 수 있게 익혀 두면 서버를 점진적으로 발전시키는 작업이 한결 수월해집니다.

In [ ]:
# Remove command preview
print("=== 등록 해제 명령 ===")
print("  claude mcp remove structural-mcp")
print()
print("# 노트북에서 자동 실행 (주석 해제):")
print("# subprocess.run(['claude', 'mcp', 'remove', 'structural-mcp'])")


## §7. 대안 방식 — 설정 파일을 직접 편집하기 (팀 공유에 유리)

명령행 인터페이스 명령을 사용하는 대신 설정 파일을 직접 편집할 수도 있습니다. 이 방식은 **팀 협업 시 매우 유용**합니다. 설정 파일이 위치할 수 있는 두 자리는 다음과 같습니다. 사용자 홈 디렉터리 안의 클로드 설정 폴더에 두면 모든 프로젝트에서 자동으로 로드되고, 프로젝트 루트에 두면 해당 프로젝트에서만 자동으로 로드됩니다.

다음은 프로젝트별 설정 파일의 예시입니다.

```json
{
  "mcpServers": {
    "structural-mcp": {
      "command": "uv",
      "args": ["run", "/Users/me/structural/structural_mcp.py"]
    }
  }
}
```

이 파일을 프로젝트 루트에 두면 **팀원이 깃 클론만 해도 자동으로 등록**되는 효과가 있습니다. 즉 본 강의에서 만든 서버를 학생들 사이에 배포할 때도 이 방식이 가장 간편하며, 팀 프로젝트의 일관된 도구 환경을 유지하는 가장 좋은 방법이기도 합니다. 명령행 인터페이스 명령은 한 번씩 실행해야 하지만, 설정 파일은 깃 저장소에 함께 들어가 있어 협업이 자연스럽게 이루어집니다.


In [ ]:
# 프로젝트 .mcp.json 자동 생성 헬퍼
import json
from pathlib import Path

py_path = Path("structural_mcp.py").resolve()
config = {
    "mcpServers": {
        "structural-mcp": {
            "command": "uv",
            "args": ["run", str(py_path)],
        }
    }
}
out = Path(".mcp.json")
out.write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved: {out.resolve()}")
print(out.read_text(encoding="utf-8"))


## §8. 6단계 빌드업 회고 — 학습 마무리

본 노트북 시리즈는 강의노트 §2.7의 6단계 빌드업을 그대로 따라왔습니다. 단계 1 노트북에서 FastMCP 인스턴스를 생성하였고, 단계 2 노트북에서 KDS 요약을 리소스로 노출하였으며, 단계 3 노트북에서 휨과 전단 검토 도구를 추가했습니다. 본 캠페인의 핵심인 단계 4 노트북에서 미다스 파서 도구를, 단계 5 노트북에서 구조 검토 프롬프트를 추가하였고, 마지막 단계 6 노트북에서 클로드 코드에 등록하고 자연어 시연까지 마쳤습니다.

각 단계의 산출물을 한자리에 모아 보면 다음과 같습니다. 도구는 휨 검토용과 전단 검토용과 미다스 파일 파싱용 세 개, 리소스는 콘크리트 설계기준 요약 한 개, 프롬프트는 일반 구조 검토용과 기둥 전용 두 개입니다. 이 구성은 강의노트가 권장하는 최소 완성형이자 한국 건축 실무에서 즉시 응용 가능한 단위 묶음이기도 합니다.

### 본 서버가 9주차 멀티 에이전트에서 갖는 의미

강의노트 §2.7 라인 1900 부근의 멀티 에이전트 청사진에서 **본 서버는 한 명의 도메인 전문 에이전트**로 동작합니다. 본 강의에서 9주차에 다루게 될 라우터 패턴은 사용자의 자연어 요청을 분석하여 어떤 도메인 에이전트에게 보낼지 자동으로 결정합니다. 그 시점에서 본 노트북에서 만든 서버는 한 명의 구조 도메인 전문가로서 다른 에이전트들과 함께 가상 연구실의 일원이 됩니다.

```
조정자 (9주차의 라우터)
  ├─ structural-mcp     ← 본 노트북 시리즈에서 직접 만든 서버
  ├─ bim-mcp            (10주차에서 다룸)
  └─ design-doc-rag-mcp (12주차에서 다룸)
```

여러 모델 컨텍스트 프로토콜 서버가 모이면 **가상의 연구실**이 됩니다. 한 번 만든 서버는 **8주차의 클로드 코드 심화, 9주차의 멀티 에이전트, 11주차의 미다스 응용** 단계에서 모두 재사용할 수 있으며, *"한 번 만들면 모든 학생과 프로젝트가 같은 해석 능력을 자연어로 공유"* 하는 효과를 누릴 수 있습니다 (강의노트 §2.7 라인 1570).

> [!finding] 도메인 모델 컨텍스트 프로토콜 서버의 진짜 가치
> 4주차의 도구 사용은 매 앱마다 도구를 중복으로 구현해야 한다는 한계가 있고, 5주차의 검색 증강 생성은 문서 검색만 가능할 뿐 계산이나 수정은 할 수 없습니다. **7주차의 모델 컨텍스트 프로토콜은 두 가지를 한 서버에 묶어 어디서든 재사용**할 수 있게 해줍니다. 이것이 본 강의가 7주차에 모델 컨텍스트 프로토콜을 핵심 주제로 다루는 이유입니다.

### 본 캠페인이 한국 건축 실무에 시사하는 바

본 캠페인의 산출물은 단순한 학습용 예시를 넘어 한국 건축 실무에서 즉시 응용 가능한 패턴을 담고 있습니다. 한국에서 가장 널리 쓰이는 미다스 시빌과 미다스 젠의 모델 파일을 직접 읽어들이는 도구, 한국설계기준을 따르는 검토 절차를 자동으로 강제하는 프롬프트, 그리고 한 번 만들면 모든 프로젝트에서 재사용할 수 있는 등록 메커니즘이 모두 한 자리에 모여 있기 때문입니다.

학생들은 본 노트북 시리즈를 마친 시점에서 본인만의 도메인 서버를 한 개씩 손에 쥐게 됩니다. 졸업 후 실무에 진입하더라도 본 서버를 계속 발전시켜 가면서 자신만의 전문 도구를 키워 갈 수 있고, 이는 곧 본 강의가 추구하는 **자기 도구를 만들 줄 아는 건축 엔지니어** 라는 목표와 정확히 맞닿아 있습니다.

---

### 다음 행보
8주차에서는 클로드 코드를 더 깊이 활용합니다. 서브에이전트와 후크와 깃 워크트리 등 고급 워크플로를 익힙니다. 9주차에서는 멀티 에이전트 라우팅을 다루며 여러 도메인 서버를 자동으로 오케스트레이션하는 방법을 배웁니다. 11주차에서는 미다스 모델 컨텍스트 프로토콜 심화 내용으로 본 노트북의 미다스 파서를 실제 해석 결과 처리까지 확장합니다. 본 노트북에서 만든 서버는 이 모든 후속 단계에서 변함없이 사용되므로, 본 단계까지의 학습이 후반부 강의 전반의 든든한 토대가 됩니다.


## §9. 한계 사항 및 주의사항

본 단계를 진행할 때 유의해야 할 한계와 주의사항을 다음과 같이 정리합니다. 학습자가 자주 부딪히는 함정들을 미리 정리해 두었으므로 한 번씩 살펴본 뒤 실습을 시작하기를 권장합니다.

**클로드 코드 의존성**: 등록 명령은 클로드 코드 명령행 인터페이스 설치를 전제로 합니다. 미설치 환경에서는 §3과 §4의 셀이 화면에 명령을 출력만 할 뿐, 실제 등록은 건너뜁니다. 학습자가 명령행 인터페이스를 아직 설치하지 않았더라도 본 노트북의 다른 셀들은 그대로 학습 자료로 활용할 수 있습니다.

**`uv` 와 표준 파이썬의 차이**: 권장 명령은 `uv run`이지만, 환경에 `uv` 가 없다면 표준 파이썬 인터프리터로 대체할 수 있습니다. 두 방식 모두 동일한 결과를 만들어내므로 본인의 환경에 맞춰 선택하면 됩니다.

**전송 방식**: 본 단계는 표준 입출력 전송 방식을 사용합니다. 하이퍼텍스트 전송 규약 등 다른 전송 방식은 11주차의 미다스 응용 단계에서 다룹니다. 표준 입출력 전송은 같은 컴퓨터에서 동작하는 서버에 가장 적합한 방식입니다.

**서버 갱신 시 재등록**: 본 노트북에서 만든 통합 파일을 수정하여 새로운 도구를 추가했다면 클로드 코드 세션을 재시작하는 것을 권장합니다. 도구 시그니처가 바뀌었을 때는 등록을 해제하고 다시 등록하는 편이 가장 안전합니다.

**절대경로 사용은 필수**: 등록 명령과 도구 인자 모두 반드시 절대경로를 사용해야 합니다. 이는 운영 규칙의 'absolute paths only' 원칙에 따른 것이며, 학생들이 가장 자주 부딪히는 실수이기도 합니다.

**API 키 처리**: 본 서버 자체는 앤트로픽 API 키가 필요하지 않습니다. 클로드 코드 측에서 사용하는 API 키만 정상적으로 설정되어 있으면 본 서버를 호출할 수 있습니다. 즉 학생들이 자신의 컴퓨터에서 본 서버를 띄워도 추가 비용 없이 활용할 수 있습니다.
